# smolagents: code as action

Most tool agents ask for one JSON tool call at a time. A code agent writes Python that can call several tools, calculate, and branch in one action.

In this notebook, we will:

- connect the course model to smolagents;
- expose the shop tools;
- compare a tool-calling agent with a code agent;
- see why code can compose work; and
- inspect the security cost of running model-written code.

Before running this notebook, install smolagents once with: pip install -e ".[smolagents]".

In [ ]:
import contextlib
import io
import os
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
TEMPERATURE = 0
litellm.drop_params = True

## 1. Create the model

LiteLLMModel uses the same provider-prefixed model name as the rest of the course.

In [ ]:
from smolagents import LiteLLMModel

model = LiteLLMModel(model_id=MODEL, temperature=TEMPERATURE)

print(type(model).__name__)
print(model.model_id)

## 2. Wrap the shop tools

The tool decorator reads the function name, type hints, and docstring to create a model-facing schema.

In [ ]:
from smolagents import tool
from shoplab import world

orders = {item["order_id"]: item for item in world.load_orders()}
customers = {item["customer_id"]: item for item in world.load_customers()}

@tool
def get_order(order_id: str) -> dict:
    """Look up one order.

    Args:
        order_id: An order id such as ORD-7312.
    """
    return orders.get(order_id, {"error": "order not found"})

@tool
def get_customer(customer_id: str) -> dict:
    """Look up one customer.

    Args:
        customer_id: A customer id such as CUST-07.
    """
    return customers.get(customer_id, {"error": "customer not found"})

@tool
def find_policy(query: str) -> list:
    """Search shop policies.

    Args:
        query: Words that describe the return question.
    """
    return world.search_policy(query)

TOOLS = [get_order, get_customer, find_policy]
print([item.name for item in TOOLS])

## 3. ToolCallingAgent uses JSON actions

This is the familiar tool loop: the model requests a named tool with arguments, receives the result, and continues.

In [ ]:
TASK = (
    "Triage TKT-2205. Order ORD-7312, customer CUST-07. The opened Torrent "
    "boots were returned after 18 days for a refund. Look up order, customer, "
    "and policy. Return decision, policy id, and refund amount."
)

In [ ]:
from smolagents import ToolCallingAgent
from smolagents.monitoring import LogLevel

tool_agent = ToolCallingAgent(
    tools=TOOLS,
    model=model,
    max_steps=6,
    verbosity_level=LogLevel.OFF,
)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    tool_answer = tool_agent.run(TASK)

for step in tool_agent.memory.steps:
    for call in getattr(step, "tool_calls", None) or []:
        print(call.name, call.arguments)

print("answer:", tool_answer)

## 4. CodeAgent uses Python actions

A CodeAgent writes a Python block. The block can call tools and use their returned values.

In [ ]:
from smolagents import CodeAgent

code_agent = CodeAgent(
    tools=TOOLS,
    model=model,
    max_steps=6,
    verbosity_level=LogLevel.OFF,
)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    code_answer = code_agent.run(TASK)

for step in code_agent.memory.steps:
    action = getattr(step, "code_action", None)
    if action:
        print("Python action:\n", action[:400])

print("answer:", code_answer)

## 5. Code can compose work

The next task needs a lookup and arithmetic. A code block can do both without a separate calculator tool.

In [ ]:
calculator_agent = CodeAgent(
    tools=[get_order],
    model=model,
    max_steps=4,
    verbosity_level=LogLevel.OFF,
)

calculation_task = (
    "For order ORD-7312 and sku LK-1016, calculate full item value and the "
    "refund after a 10 percent fee. Return a dict with full and after_fee."
)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    calculation = calculator_agent.run(calculation_task)

code_blocks = [
    step.code_action
    for step in calculator_agent.memory.steps
    if getattr(step, "code_action", None)
]

print("code blocks:", len(code_blocks))
print(code_blocks[-1])
print("result:", calculation)

## 6. The executor is a security boundary

CodeAgent runs model-written code. The local executor blocks many operations, but local execution still deserves the same caution as any untrusted code.

In [ ]:
from smolagents.local_python_executor import LocalPythonExecutor

executor = LocalPythonExecutor(additional_authorized_imports=[])

def try_code(text):
    try:
        return executor(text)
    except Exception as error:
        return str(error)[:160]

print("math:", try_code("189.99 * 0.9"))
print("import:", try_code("import os\nos.getcwd()"))
print("file:", try_code("open('/etc/passwd').read()"))

For stronger isolation, run model-written code in a separate sandbox such as a container or remote execution environment. Allow only the imports and data the task needs.

Use tool calls when actions are small and easy to audit. Code as action can help when one step needs lookup, filtering, arithmetic, and branching. Its flexibility comes with a larger security surface.

## What to remember

1. ToolCallingAgent emits named JSON tool calls.
2. CodeAgent emits Python actions.
3. Code can combine several operations in one step.
4. The executor becomes part of the security boundary.
5. Limit imports, files, network, and secrets.
6. Choose the action space that fits the work and risk.

## Small practice steps

1. Count tool calls made by ToolCallingAgent.
2. Ask CodeAgent to filter two order items.
3. Try one more blocked operation in LocalPythonExecutor.